# Sesion 5 — T4: Funciones Avanzadas y Modulos
## Diplomado: Machine Learning en Seguros
### Facultad de Ciencias, UNAM · 24 de abril de 2026  (18:00 - 21:00 h)

---

**Instructor:** Eric   **Sesion:** 5 de 13

---

> **Objetivo:** Completar el tema de funciones con argumentos variables (*args, **kwargs),
> aprender a proteger funciones de datos sucios (try/except),
> y entender como funcionan los modulos de Python.

## Contenido

1. [*args — argumentos posicionales variables](#1-args)
2. [**kwargs — argumentos con nombre variables](#2-kwargs)
3. [try/except — proteger funciones de errores](#3-try)
4. [Errores comunes en datos de seguros](#4-errores)
5. [Que es un modulo y como importar](#5-modulos)
6. [Modulo math — funciones matematicas](#6-math)
7. [Modulo datetime en funciones](#7-datetime)
8. [Ejercicio integrador de cierre](#8-ejercicio)

---
<a id='1-args'></a>
## 1. *args — Argumentos Posicionales Variables

**El problema:** quieres una funcion que sume primas,
pero no siempre sabes cuantas polizas llegaran.

**La solucion:** el `*` antes del nombre del parametro le dice a Python
*'acepta cualquier cantidad de argumentos y empacarlos en una tupla'*.

In [1]:
# ── El problema sin *args ────────────────────────────────────────────────────
# Tendrias que definir una funcion por cada caso posible
def sumar_2(p1, p2):         return p1 + p2
def sumar_3(p1, p2, p3):     return p1 + p2 + p3
def sumar_4(p1, p2, p3, p4): return p1 + p2 + p3 + p4
# ... esto no escala

# ── La solucion con *args ────────────────────────────────────────────────────
def sumar_primas(*primas):
    # primas llega como una TUPLA
    print(f'Tipo: {type(primas)}')
    print(f'Valores: {primas}')
    return sum(primas)

print(sumar_primas(2400))
print(sumar_primas(2400, 6200, 1900))
print(sumar_primas(2400, 6200, 1900, 14500, 3200))

Tipo: <class 'tuple'>
Valores: (2400,)
2400
Tipo: <class 'tuple'>
Valores: (2400, 6200, 1900)
10500
Tipo: <class 'tuple'>
Valores: (2400, 6200, 1900, 14500, 3200)
28200


In [2]:
# ── Iterar sobre *args ───────────────────────────────────────────────────────
def mostrar_cartera(*primas):
    print(f'Cartera: {len(primas)} polizas')
    for i, prima in enumerate(primas, 1):
        print(f'  Poliza {i}: ${prima:>8,.2f}')
    print(f'  Total:   ${sum(primas):>8,.2f}')
    print(f'  Promedio:${sum(primas)/len(primas):>8,.2f}')

mostrar_cartera(2400, 6200, 1900, 14500, 3200, 8800)

Cartera: 6 polizas
  Poliza 1: $2,400.00
  Poliza 2: $6,200.00
  Poliza 3: $1,900.00
  Poliza 4: $14,500.00
  Poliza 5: $3,200.00
  Poliza 6: $8,800.00
  Total:   $37,000.00
  Promedio:$6,166.67


In [3]:
# ── Combinar parametros normales con *args ───────────────────────────────────
# Regla: los parametros normales van PRIMERO, *args al final

def reporte_ramo(ramo, *primas):
    if len(primas) == 0:
        print(f'{ramo}: sin polizas')
        return
    print(f'Ramo: {ramo}')
    print(f'  Polizas:  {len(primas)}')
    print(f'  Total:    ${sum(primas):,.2f}')
    print(f'  Promedio: ${sum(primas)/len(primas):,.2f}')

reporte_ramo('GMM',   2400, 6200, 1900, 14500)
reporte_ramo('Autos', 5800, 3200, 7100)
reporte_ramo('Vida')   # sin primas

Ramo: GMM
  Polizas:  4
  Total:    $25,000.00
  Promedio: $6,250.00
Ramo: Autos
  Polizas:  3
  Total:    $16,100.00
  Promedio: $5,366.67
Vida: sin polizas


In [4]:
# ── *args con calculos estadisticos actuariales ──────────────────────────────
import math

def estadisticas(*primas):
    """Calcula metricas de una cartera de primas.
    
    Parametros
    ----------
    *primas : float
        Primas anuales de la cartera.
    
    Retorna
    -------
    dict con n, total, promedio, maximo, minimo, desv_std
    """
    if not primas:
        return None
    n     = len(primas)
    total = sum(primas)
    prom  = total / n
    varianza = sum((p - prom)**2 for p in primas) / n
    desv_std = math.sqrt(varianza)
    return dict(n=n, total=total, promedio=prom,
                maximo=max(primas), minimo=min(primas),
                desv_std=desv_std)

res = estadisticas(2400, 6200, 1900, 14500, 3200, 8800, 1850, 7300)
for k, v in res.items():
    print(f'  {k:<10}: {v:>10,.2f}')

  n         :       8.00
  total     :  46,150.00
  promedio  :   5,768.75
  maximo    :  14,500.00
  minimo    :   1,850.00
  desv_std  :   4,126.51


---
<a id='2-kwargs'></a>
## 2. **kwargs — Argumentos con Nombre Variables

Mientras `*args` captura argumentos posicionales (sin nombre),
`**kwargs` captura argumentos pasados como `nombre=valor`.
Dentro de la funcion llega como un **diccionario**.

```
# kwargs = keyword arguments
def mi_func(**datos):
    # datos es un dict
```

In [5]:
# ── **kwargs basico ──────────────────────────────────────────────────────────
def crear_poliza(**campos):
    print(f'Tipo: {type(campos)}')
    print('Campos recibidos:')
    for clave, valor in campos.items():
        print(f'  {clave}: {valor}')

crear_poliza(
    nombre='Ana Torres',
    suma=500_000,
    ramo='GMM',
    activa=True,
    deducible=5000
)

Tipo: <class 'dict'>
Campos recibidos:
  nombre: Ana Torres
  suma: 500000
  ramo: GMM
  activa: True
  deducible: 5000


In [6]:
# ── Aplicacion practica: funcion de reporte flexible ────────────────────────
def imprimir_registro(titulo, **campos):
    """Imprime un registro con cualquier cantidad de campos."""
    print(f'--- {titulo} ---')
    for k, v in campos.items():
        # Formatear numeros con separador de miles
        if isinstance(v, (int, float)):
            print(f'  {k:<18}: {v:>12,.2f}')
        else:
            print(f'  {k:<18}: {v}')

imprimir_registro('Poliza GMM-001',
    titular='Ana Torres',
    suma_asegurada=500_000,
    prima_anual=2400,
    deducible=5000,
    activa=True
)

--- Poliza GMM-001 ---
  titular           : Ana Torres
  suma_asegurada    :   500,000.00
  prima_anual       :     2,400.00
  deducible         :     5,000.00
  activa            :         1.00


In [7]:
# ── *args y **kwargs juntos ──────────────────────────────────────────────────
# Orden: obligatorios → *args → **kwargs

def registrar_cartera(ramo, *primas, **opciones):
    """Registra una cartera con primas y opciones adicionales."""
    print(f'Ramo: {ramo}')
    print(f'Primas: {primas}')
    print(f'Total: ${sum(primas):,.2f}')
    if opciones:
        print('Opciones:')
        for k, v in opciones.items():
            print(f'  {k}: {v}')

registrar_cartera(
    'GMM',
    2400, 6200, 1900,     # *primas
    agente='Carlos R.',   # **opciones
    zona='CDMX',
    vigencia='Anual'
)

Ramo: GMM
Primas: (2400, 6200, 1900)
Total: $10,500.00
Opciones:
  agente: Carlos R.
  zona: CDMX
  vigencia: Anual


---
<a id='3-try'></a>
## 3. try/except — Proteger Funciones de Errores

Los datos reales nunca son perfectos. Un solo dato malo puede detener
el procesamiento de miles de registros. `try/except` evita eso.

```
try:
    # codigo que puede fallar
except TipoDeError:
    # que hacer si falla
finally:
    # se ejecuta SIEMPRE (opcional)
```

In [8]:
# ── Sin try/except: un error detiene todo ────────────────────────────────────
def frecuencia_sin_proteccion(siniestros, exposicion):
    return siniestros / exposicion

# Esto funciona
print(frecuencia_sin_proteccion(3, 365))

# Esto ROMPE el programa
try:
    frecuencia_sin_proteccion(3, 0)   # ZeroDivisionError
except Exception as e:
    print(f'Error sin proteccion: {e}')

# ── Con try/except: maneja y continua ────────────────────────────────────────
def frecuencia(siniestros, exposicion):
    """Calcula frecuencia con manejo de errores."""
    try:
        return siniestros / exposicion
    except ZeroDivisionError:
        return 0.0    # exposicion cero: frecuencia cero
    except TypeError:
        return None   # dato invalido

print(frecuencia(3, 365))   # normal
print(frecuencia(3, 0))     # ZeroDivisionError → 0.0
print(frecuencia(3, None))  # TypeError → None

0.00821917808219178
Error sin proteccion: division by zero
0.00821917808219178
0.0
None


In [9]:
# ── Capturar el mensaje de error con 'as e' ─────────────────────────────────
def convertir_prima(valor):
    """Convierte un valor a float. Retorna None si no es posible."""
    try:
        return float(valor)
    except (ValueError, TypeError) as e:
        print(f'  Advertencia: no se pudo convertir {repr(valor)} → {e}')
        return None

valores_csv = ['3450.75', '6200', 'N/A', '', None, '1,900']
print('Procesando primas del CSV:')
for v in valores_csv:
    resultado = convertir_prima(v)
    if resultado is not None:
        print(f'  {repr(v):15} → ${resultado:,.2f}')

Procesando primas del CSV:
  '3450.75'       → $3,450.75
  '6200'          → $6,200.00
  Advertencia: no se pudo convertir 'N/A' → could not convert string to float: 'N/A'
  Advertencia: no se pudo convertir '' → could not convert string to float: ''
  Advertencia: no se pudo convertir None → float() argument must be a string or a real number, not 'NoneType'
  Advertencia: no se pudo convertir '1,900' → could not convert string to float: '1,900'


In [10]:
# ── finally: codigo que siempre se ejecuta ───────────────────────────────────
# Util para liberar recursos, cerrar archivos, registrar log

def procesar_con_log(datos, log=[]):
    """Procesa datos y registra en log si hay error."""
    try:
        prima = float(datos['prima'])
        exp   = int(datos['exposicion'])
        resultado = prima / exp
        log.append(f"OK: {datos['id']}")
        return resultado
    except (ValueError, ZeroDivisionError, KeyError) as e:
        log.append(f"ERROR: {datos.get('id','?')} - {e}")
        return None
    finally:
        pass   # aqui podrias cerrar una conexion a BD, etc.

mi_log = []
registros = [
    {'id':'P01', 'prima':'3200', 'exposicion':'365'},
    {'id':'P02', 'prima':'N/A',  'exposicion':'180'},
    {'id':'P03', 'prima':'5800', 'exposicion':'0'},
]
for r in registros:
    procesar_con_log(r, mi_log)

print('Log del proceso:')
for entrada in mi_log:
    print(f'  {entrada}')

Log del proceso:
  OK: P01
  ERROR: P02 - could not convert string to float: 'N/A'
  ERROR: P03 - float division by zero


---
<a id='4-errores'></a>
## 4. Errores Comunes en Datos de Seguros

| Error | Causa tipica en seguros | Solucion |
|-------|------------------------|----------|
| `ZeroDivisionError` | Exposicion cero, periodo sin polizas | Retornar 0 o None |
| `ValueError` | Prima como texto ('N/A', '') | float() con try/except |
| `TypeError` | Campo None en CSV | Validar antes de operar |
| `KeyError` | Columna no existe en dict/DataFrame | .get() con default |

In [11]:
# ── Funcion robusta que combina todo ─────────────────────────────────────────
from datetime import date

def procesar_poliza(datos):
    """
    Procesa un registro de poliza con manejo completo de errores.
    Retorna dict con campos calculados y flag 'ok'.
    """
    try:
        # Convertir y validar
        prima      = float(datos['prima'])
        exposicion = int(datos['exposicion'])
        siniestros = int(datos['siniestros'])

        # Validacion de negocio
        if prima <= 0:
            raise ValueError(f'Prima invalida: {prima}')
        if exposicion <= 0:
            raise ValueError(f'Exposicion invalida: {exposicion}')

        frecuencia = siniestros / exposicion
        prima_dia  = prima / exposicion

        return {
            'id':         datos['id'],
            'prima':      prima,
            'frecuencia': round(frecuencia, 6),
            'prima_dia':  round(prima_dia, 4),
            'ok':         True,
            'error':      None
        }
    except (ValueError, KeyError, TypeError, ZeroDivisionError) as e:
        return {
            'id':    datos.get('id', '?'),
            'ok':    False,
            'error': str(e)
        }

# Cartera con datos sucios
cartera = [
    {'id':'P01','prima':'3200','exposicion':'365','siniestros':'1'},
    {'id':'P02','prima':'N/A', 'exposicion':'180','siniestros':'0'},
    {'id':'P03','prima':'5800','exposicion':'0',  'siniestros':'2'},
    {'id':'P04','prima':'9100','exposicion':'365','siniestros':'0'},
    {'id':'P05','prima':'-500','exposicion':'200','siniestros':'1'},
]

print(f'{"ID":<6} {"Prima":>8} {"Frecuencia":>12} {"Estado"}')
print('-' * 45)
ok_count = err_count = 0
for r in cartera:
    res = procesar_poliza(r)
    if res['ok']:
        ok_count += 1
        print(f"{res['id']:<6} ${res['prima']:>7,.0f} {res['frecuencia']:>12.6f}  OK")
    else:
        err_count += 1
        print(f"{res['id']:<6} {'---':>8} {'---':>12}  ERROR: {res['error']}")
print(f'Procesados: {ok_count} OK, {err_count} errores de {len(cartera)} total')

ID        Prima   Frecuencia Estado
---------------------------------------------
P01    $  3,200     0.002740  OK
P02         ---          ---  ERROR: could not convert string to float: 'N/A'
P03         ---          ---  ERROR: Exposicion invalida: 0
P04    $  9,100     0.000000  OK
P05         ---          ---  ERROR: Prima invalida: -500.0
Procesados: 2 OK, 3 errores de 5 total


---
<a id='5-modulos'></a>
## 5. Que es un Modulo y Como Importar

Un **modulo** es un archivo `.py` con funciones, clases y variables.
Permite organizar y reutilizar codigo entre proyectos.

| Tipo | De donde viene | Como se usa |
|------|---------------|-------------|
| Estandar | Incluido con Python | `import math` |
| Terceros | `pip install` | `import pandas as pd` |
| Propio | Tu archivo .py | `import mi_modulo` |

In [ ]:
# ── Formas de importar ───────────────────────────────────────────────────────

# Forma 1: importar el modulo completo
import math
print(math.sqrt(16))      # acceder con math.funcion()

# Forma 2: importar solo lo que necesitas
from math import sqrt, log, pi
print(sqrt(16))           # sin el prefijo math.
print(round(pi, 4))

# Forma 3: importar con alias (nombre corto)
import math as m
print(m.sqrt(16))

# Forma 4: importar todo (NO recomendado — contamina el namespace)
# from math import *   # evitar esto

# Ver que contiene un modulo
print([x for x in dir(math) if not x.startswith('_')])

---
<a id='6-math'></a>
## 6. Modulo math — Funciones Matematicas

El modulo mas util para calculos actuariales junto con `statistics`.

In [12]:
import math

# ── Funciones basicas ────────────────────────────────────────────────────────
print('Raiz cuadrada:  ', math.sqrt(144))
print('Logaritmo nat: ', round(math.log(1000), 4))
print('Log base 10:   ', math.log10(1000))
print('Exponencial:   ', round(math.exp(1), 5))
print('Potencia:      ', math.pow(2, 10))
print('Valor absoluto:', math.fabs(-45.3))
print('Piso:          ', math.floor(3.9))
print('Techo:         ', math.ceil(3.1))

# Constantes
print(f'Pi  = {math.pi:.10f}')
print(f'e   = {math.e:.10f}')
print(f'inf = {math.inf}')

Raiz cuadrada:   12.0
Logaritmo nat:  6.9078
Log base 10:    3.0
Exponencial:    2.71828
Potencia:       1024.0
Valor absoluto: 45.3
Piso:           3
Techo:          4
Pi  = 3.1415926536
e   = 2.7182818285
inf = inf


In [13]:
import math

# ── Aplicaciones actuariales con math ────────────────────────────────────────

# 1. Valor presente de una renta vencida
def valor_presente_renta(flujo, tasa_anual, n_anios):
    """Calcula el valor presente de flujos periodicos."""
    if tasa_anual == 0:
        return flujo * n_anios
    return flujo * (1 - math.pow(1 + tasa_anual, -n_anios)) / tasa_anual

vp = valor_presente_renta(10_000, 0.08, 10)
print(f'VP renta $10,000 / 10 anios / 8%: ${vp:,.2f}')

# 2. Volatilidad anualizada
def anualizar_volatilidad(vol_diaria, dias_habiles=252):
    """Convierte volatilidad diaria a anual."""
    return vol_diaria * math.sqrt(dias_habiles)

print(f'Vol anual: {anualizar_volatilidad(0.015):.4f}')

# 3. Desviacion estandar de primas
def desv_std(valores):
    """Desviacion estandar poblacional."""
    n    = len(valores)
    media = sum(valores) / n
    return math.sqrt(sum((x - media)**2 for x in valores) / n)

primas = [2400, 6200, 1900, 14500, 3200, 8800]
print(f'Media:   ${sum(primas)/len(primas):,.2f}')
print(f'Desv Std:${desv_std(primas):,.2f}')
print(f'CV:      {desv_std(primas)/(sum(primas)/len(primas)):.4f}')

VP renta $10,000 / 10 anios / 8%: $67,100.81
Vol anual: 0.2381
Media:   $6,166.67
Desv Std:$4,426.69
CV:      0.7178


---
<a id='7-datetime'></a>
## 7. Modulo datetime en Funciones

Ya usamos `datetime` en S2. Ahora lo encapsulamos en funciones
reutilizables para calculos de vigencia y aging.

In [14]:
from datetime import date, timedelta

# ── Funciones de vigencia ─────────────────────────────────────────────────────
def dias_vigencia(emision, vencimiento):
    """Dias totales entre emision y vencimiento."""
    return (vencimiento - emision).days

def fraccion_expuesta(emision, vencimiento, referencia=None):
    """Fraccion del periodo vigente a la fecha de referencia."""
    if referencia is None:
        referencia = date.today()
    total    = (vencimiento - emision).days
    trans    = (referencia - emision).days
    if total <= 0: return 0.0
    return round(min(max(trans / total, 0.0), 1.0), 4)

def proxima_renovacion(vencimiento):
    """Fecha de inicio de la siguiente vigencia."""
    return vencimiento + timedelta(days=1)

# Probar
emision = date(2026, 1, 15)
vence   = date(2026, 12, 31)
print(f'Dias de vigencia:   {dias_vigencia(emision, vence)}')
print(f'Fraccion expuesta:  {fraccion_expuesta(emision, vence):.1%}')
print(f'Proxima renovacion: {proxima_renovacion(vence)}')

Dias de vigencia:   350
Fraccion expuesta:  30.3%
Proxima renovacion: 2027-01-01


In [15]:
from datetime import date

# ── Aging de siniestros ───────────────────────────────────────────────────────
def clasificar_aging(fecha_siniestro, referencia=None):
    """Clasifica un siniestro por antiguedad para reservas IBNR.
    
    Retorna el bucket de aging segun los dias transcurridos.
    """
    if referencia is None:
        referencia = date.today()
    dias = (referencia - fecha_siniestro).days
    if dias <= 30:    return '0-30 dias'
    elif dias <= 90:  return '31-90 dias'
    elif dias <= 180: return '91-180 dias'
    elif dias <= 365: return '181-365 dias'
    else:             return 'Mas de 1 anio'

# Cartera de siniestros
siniestros = [
    ('SIN-001', date(2026, 4, 1),  45_000),
    ('SIN-002', date(2026, 2, 10), 92_000),
    ('SIN-003', date(2025, 11, 5), 28_000),
    ('SIN-004', date(2025, 3, 20), 380_000),
]

print(f'{"ID":<10} {"Fecha":<14} {"Monto":>10} {"Bucket"}')
print('-' * 55)
for id_, fecha, monto in siniestros:
    bucket = clasificar_aging(fecha)
    print(f'{id_:<10} {str(fecha):<14} ${monto:>9,} {bucket}')

ID         Fecha               Monto Bucket
-------------------------------------------------------
SIN-001    2026-04-01     $   45,000 0-30 dias
SIN-002    2026-02-10     $   92,000 31-90 dias
SIN-003    2025-11-05     $   28,000 91-180 dias
SIN-004    2025-03-20     $  380,000 Mas de 1 anio


---
<a id='8-ejercicio'></a>
## 8. Ejercicio Integrador de Cierre

Combina todo lo visto en esta sesion:
`*args`, `**kwargs`, `try/except`, `math` y `datetime`.

In [16]:
# ── Datos con errores reales ─────────────────────────────────────────────────
from datetime import date
import math

registros = [
    {'id':'A-01','prima':'3200','exposicion':'365','siniest':'1','emision':'2026-01-15'},
    {'id':'A-02','prima':'N/A', 'exposicion':'180','siniest':'0','emision':'2026-03-01'},
    {'id':'A-03','prima':'5800','exposicion':'0',  'siniest':'2','emision':'2025-07-01'},
    {'id':'A-04','prima':'9100','exposicion':'365','siniest':'0','emision':'2026-02-10'},
    {'id':'A-05','prima':'4200','exposicion':'270','siniest':'1','emision':'2025-09-15'},
]
print(f'{len(registros)} registros cargados')

5 registros cargados


In [17]:
# ── Tarea 1: procesar cada registro con try/except ───────────────────────────
# Escribe una funcion procesar(datos) que:
# - Convierta prima a float y exposicion a int
# - Calcule frecuencia (siniest / exposicion)
# - Calcule prima_dia (prima / exposicion)
# - Devuelva dict con los campos calculados y ok=True/False

def procesar(datos):
    try:
        prima = float(datos['prima'])
        exp   = int(datos['exposicion'])
        sin   = int(datos['siniest'])
        if exp <= 0: raise ValueError('exposicion <= 0')
        return {'id':datos['id'], 'prima':prima,
                'frecuencia':sin/exp, 'prima_dia':prima/exp, 'ok':True}
    except (ValueError, KeyError, TypeError, ZeroDivisionError) as e:
        return {'id':datos.get('id','?'), 'ok':False, 'error':str(e)}

In [18]:
# ── Tarea 2: funcion de resumen con *args ────────────────────────────────────
# Escribe resumen_cartera(*resultados) que reciba los dicts del paso anterior
# y calcule: total OK, total errores, prima promedio (solo validos),
#            desv std de primas (usa math.sqrt), frecuencia promedio

def resumen_cartera(*resultados):
    validos = [r for r in resultados if r['ok']]
    errores = [r for r in resultados if not r['ok']]
    if not validos: return None
    primas = [r['prima'] for r in validos]
    media  = sum(primas) / len(primas)
    desv   = math.sqrt(sum((p-media)**2 for p in primas) / len(primas))
    frec_p = sum(r['frecuencia'] for r in validos) / len(validos)
    return dict(total=len(resultados), ok=len(validos), errores=len(errores),
                prima_prom=media, prima_desv=desv, frec_prom=frec_p)

In [19]:
# ── Tarea 3: reporte final con **kwargs ──────────────────────────────────────
# Escribe imprimir_reporte(titulo, **metricas) que reciba las metricas
# del paso anterior y las imprima con formato alineado

def imprimir_reporte(titulo, **metricas):
    print(f'--- {titulo} ---')
    for k, v in metricas.items():
        if isinstance(v, float): print(f'  {k:<15}: {v:>10.4f}')
        else:                    print(f'  {k:<15}: {v:>10}')

res = [procesar(r) for r in registros]
metricas = resumen_cartera(*res)
imprimir_reporte('Cartera Autos Abril 2026', **metricas)


--- Cartera Autos Abril 2026 ---
  total          :          5
  ok             :          3
  errores        :          2
  prima_prom     :  5500.0000
  prima_desv     :  2578.1130
  frec_prom      :     0.0021


---
## Soluciones (descomenta para revelar)


In [ ]:
# ── Solucion completa ────────────────────────────────────────────────────────

# def procesar(datos):
#     try:
#         prima = float(datos['prima'])
#         exp   = int(datos['exposicion'])
#         sin   = int(datos['siniest'])
#         if exp <= 0: raise ValueError('exposicion <= 0')
#         return {'id':datos['id'], 'prima':prima,
#                 'frecuencia':sin/exp, 'prima_dia':prima/exp, 'ok':True}
#     except (ValueError, KeyError, TypeError, ZeroDivisionError) as e:
#         return {'id':datos.get('id','?'), 'ok':False, 'error':str(e)}
#
# def resumen_cartera(*resultados):
#     validos = [r for r in resultados if r['ok']]
#     errores = [r for r in resultados if not r['ok']]
#     if not validos: return None
#     primas = [r['prima'] for r in validos]
#     media  = sum(primas) / len(primas)
#     desv   = math.sqrt(sum((p-media)**2 for p in primas) / len(primas))
#     frec_p = sum(r['frecuencia'] for r in validos) / len(validos)
#     return dict(total=len(resultados), ok=len(validos), errores=len(errores),
#                 prima_prom=media, prima_desv=desv, frec_prom=frec_p)
#
# def imprimir_reporte(titulo, **metricas):
#     print(f'--- {titulo} ---')
#     for k, v in metricas.items():
#         if isinstance(v, float): print(f'  {k:<15}: {v:>10.4f}')
#         else:                    print(f'  {k:<15}: {v:>10}')
#
# # Ejecutar
# res = [procesar(r) for r in registros]
# metricas = resumen_cartera(*res)
# imprimir_reporte('Cartera Autos Abril 2026', **metricas)

---
## Resumen de la Sesion 5

| Concepto | Lo que aprendimos |
|---------|------------------|
| **`*args`** | Cualquier cantidad de argumentos posicionales → llega como tupla |
| **`**kwargs`** | Cualquier cantidad de argumentos con nombre → llega como dict |
| **Orden** | `def f(normales, *args, **kwargs)` siempre en ese orden |
| **`try/except`** | Proteger funciones de datos sucios sin detener el proceso |
| **Errores comunes** | ZeroDivisionError, ValueError, TypeError, KeyError |
| **Modulos** | Estandar, terceros y propios — tres formas de importar |
| **`math`** | sqrt, log, exp, pow, ceil, floor, pi, e |
| **`datetime`** | En funciones: vigencias, fracciones de exposicion, aging |

**Proxima sesion — Sab 25 de abril, 07:00-11:00 h:**
Modulos propios (`mi_modulo.py`, `tabla_tarifas.py`) + inicio de Pandas.

**Tarea:**
```bash
git add sesion5_M1_notebook.ipynb
git commit -m "Sesion 5: args kwargs try/except modulos"
git push
```

---
*Diplomado ML en Seguros · Facultad de Ciencias, UNAM · 2026*